<a href="https://githubtocolab.com/tannialhernandez/topicosAvanzadosAnalitica/blob/main/E3-SongEmbeddingVisualization/E3-SongEmbeddingsVisualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Song Embeddings - Skipgram Recommender

In this notebook, we'll use human-made music playlists to learn song embeddings. We'll treat a playlist as if it's a sentence and the songs it contains as words. We feed that to the word2vec algorithm which then learns embeddings for every song we have. These embeddings can then be used to recommend similar songs. This technique is used by Spotify, AirBnB, Alibaba, and others. It accounts for a vast portion of their user activity, user media consumption, and/or sales (in the case of Alibaba).

The [dataset we'll use](https://www.cs.cornell.edu/~shuochen/lme/data_page.html) was collected by Shuo Chen from Cornell University. The dataset contains playlists from hundreds of radio stations from around the US.

## Importing packages and dataset

The following code cell uninstalls the current version of NumPy and installs NumPy 1.23.5, which is compatible with Gensim. Before that, Colab displays a warning telling you that you must restart the session in order to use the installed versions. This is because the installed version of NumPy is compatible with Python 3.11.

In [17]:
!pip uninstall -y numpy pandas gensim scipy matplotlib
!pip install --no-cache-dir "numpy==1.23.5" "pandas==2.2.2" "gensim==4.3.3" "scipy==1.13.1" "matplotlib==3.10.0"

Found existing installation: numpy 1.23.5
Uninstalling numpy-1.23.5:
  Successfully uninstalled numpy-1.23.5
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
Found existing installation: gensim 4.3.3
Uninstalling gensim-4.3.3:
  Successfully uninstalled gensim-4.3.3
Found existing installation: nltk 3.9.1
Uninstalling nltk-3.9.1:
  Successfully uninstalled nltk-3.9.1
Found existing installation: scipy 1.13.1
Uninstalling scipy-1.13.1:
  Successfully uninstalled scipy-1.13.1
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 107.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 197.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 185.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 252.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [1]:
import numpy as np
import pandas as pd
import gensim
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from gensim.models import Word2Vec
from urllib import request
import warnings
import plotly.graph_objects as go
warnings.filterwarnings('ignore')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


The playlist dataset is a text file where every line represents a playlist. That playlist is basically a series of song IDs.

In [2]:
# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]


The `playlists` variable now contains a python list. Each item in this list is a playlist containing song ids. We can look at the first two playlists here:

In [3]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

## Training the Word2Vec Model
Our dataset is now in the shape the the Word2Vec model expects as input. We pass the dataset to the model.

In [4]:
model = Word2Vec(playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4)

The model is now trained. Every song has an embedding. We only have song IDs, though, no titles or other info. Let's grab the song information file.

## Song Title and Artist File
Let's load and parse the file containing song titles and artists

In [27]:
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]

In [28]:
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df['id'] = songs_df['id'].str.strip()
songs_df = songs_df.set_index('id')

In [29]:
display("Shape of the DataFrame:", songs_df.shape)
display(songs_df.head())

'Shape of the DataFrame:'

(75263, 2)

,title,artist
id,,
0,Gucci Time (w\/ Swizz Beatz),Gucci Mane
1,Aston Martin Music (w\/ Drake & Chrisette Mich...,Rick Ross
2,Get Back Up (w\/ Chris Brown),T.I.
3,Hot Toddy (w\/ Jay-Z & Ester Dean),Usher
4,Whip My Hair,Willow


### Exercise:

Build visualization for the embeddings of the song recommender.

In [152]:
#With the increase in the amount of data, it becomes more and more difficult to visualize and interpret
#In practice, similar words are combined into groups for further visualization.
seed=25
random_indices = songs_df.sample(n=5, random_state=seed).index
random_indices_songs = songs_df.loc[random_indices]["title"].values

keys = random_indices # Songs to compare
embedding_clusters = []
word_clusters = []

for word in keys:
    embeddings = []
    words = []
    for similar_word, _ in model.wv.most_similar(word, topn=12):
        words.append(songs_df.loc[similar_word]["title"])
        embeddings.append(model.wv[similar_word])
    embedding_clusters.append(embeddings)#apending access vector of all similar words
    word_clusters.append(words)#appending list of all smiliar words

In [147]:
def tsne_plot_similar_words_3d(labels, embedding_clusters, word_clusters, a=0.7):
    fig = go.Figure()

    for label, embeddings, words in zip(labels, embedding_clusters, word_clusters):
        x = embeddings[:, 0]
        y = embeddings[:, 1]
        z = embeddings[:, 2]

        # Add points
        fig.add_trace(go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers+text',
            name=label,
            text=words,  # Labels
            textposition='top center',
            opacity=a,
            marker=dict(size=4)
        ))

    fig.update_layout(
        title='3D t-SNE Plot of Similar Words',
        width=1400,
        height=1120,
        margin=dict(l=0, r=0, b=0, t=40),
        scene=dict(
            xaxis_title='Component 1',
            yaxis_title='Component 2',
            zaxis_title='Component 3'
        )
    )

    fig.show()


In [153]:
embedding_clusters = np.array(embedding_clusters)
n, m, k = embedding_clusters.shape #geting the dimensions

In [ ]:
tsne_model_en_3d = TSNE(perplexity=5, n_components=3, init='pca', n_iter=1500, random_state=seed)
embeddings_en_3d = np.array(tsne_model_en_3d.fit_transform(embedding_clusters.reshape(n * m, k))).reshape(n, m, 3)
tsne_plot_similar_words_3d(random_indices_songs, embeddings_en_3d, word_clusters)

In [ ]:
tsne_model_en_3d = TSNE(perplexity=8, n_components=3, init='pca', n_iter=1500, random_state=seed)
embeddings_en_3d = np.array(tsne_model_en_3d.fit_transform(embedding_clusters.reshape(n * m, k))).reshape(n, m, 3)
tsne_plot_similar_words_3d(random_indices_songs, embeddings_en_3d, word_clusters)

In [ ]:
tsne_model_en_3d = TSNE(perplexity=10, n_components=3, init='pca', n_iter=1500, random_state=seed)
embeddings_en_3d = np.array(tsne_model_en_3d.fit_transform(embedding_clusters.reshape(n * m, k))).reshape(n, m, 3)
tsne_plot_similar_words_3d(random_indices_songs, embeddings_en_3d, word_clusters)

In this case, the plots with perplexity 8 and 10 are much better defined than the one with 5.